# 📊 Analyse BI : Impact des Événements Géopolitiques sur les Crypto-actifs

**Auteur** : [Ton Nom]  
**Date** : Mars 2025  
**Version** : 1.0

---

## 🎯 Objectif du notebook

Ce notebook contient le **pipeline complet** de traitement des données pour le projet d'analyse BI :

1. ✅ Collecte des données crypto via API Yahoo Finance (yfinance)
2. ✅ Nettoyage et normalisation
3. ✅ Construction de la table des événements géopolitiques
4. ✅ Feature engineering (11 variables calculées)
5. ✅ Création de la table calendrier enrichie
6. ✅ Export pour Power BI (Excel)

---

## 📦 Données produites

- **dataset_powerbi_clean.xlsx** : 8 768 lignes × 38 colonnes
- **evenements_geopolitiques.csv** : 30 événements majeurs (2020-2025)
- **grandes_puissances.xlsx** : Classification USA/Chine/Europe/Moyen-Orient
- **calendrier_enrichi.csv** : Table calendrier avec fenêtres ±30j

---

## 📚 Imports et configuration

In [ ]:
# Imports standards
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Import yfinance pour la collecte de données
import yfinance as yf

# Configuration pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Création du dossier de sortie
output_folder = "crypto_data"
os.makedirs(output_folder, exist_ok=True)

print("✅ Bibliothèques importées avec succès")
print(f"📁 Dossier de sortie : {output_folder}/")

---

# 1️⃣ COLLECTE DES DONNÉES CRYPTO

## 📥 Téléchargement via Yahoo Finance API

**Source** : Yahoo Finance (via bibliothèque yfinance)  
**Période** : 2020-01-01 → 2026-01-01 (6 années)  
**Cryptos** : BTC-USD, ETH-USD, DOGE-USD, USDC-USD  
**Format** : OHLCV (Open, High, Low, Close, Volume)  

In [ ]:
# Configuration de la collecte
tickers = ["BTC-USD", "ETH-USD", "DOGE-USD", "USDC-USD"]
start_date = "2020-01-01"
end_date = "2026-01-01"

print(f"📊 Cryptos à télécharger : {tickers}")
print(f"📅 Période : {start_date} → {end_date}")
print("\n⏳ Téléchargement en cours...\n")

In [ ]:
# Téléchargement individuel de chaque crypto
# (télécharger 1 par 1 évite les problèmes de multi-index)

dataframes = {}

for ticker in tickers:
    print(f"⏳ Téléchargement de {ticker}...")
    
    try:
        # Téléchargement
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        
        # Vérification que les données ne sont pas vides
        if data.empty:
            print(f"   ⚠️  Aucune donnée pour {ticker}")
            continue
        
        # Reset index pour avoir Date en colonne
        data = data.reset_index()
        
        # Nettoyage des colonnes multi-index si présentes
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        
        # Conversion de la date
        data['Date'] = pd.to_datetime(data['Date'])
        
        # Ajout du nom de la crypto
        data['Crypto'] = ticker.replace("-USD", "")
        
        # Sélection des colonnes nécessaires
        colonnes_necessaires = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
        colonnes_presentes = [col for col in colonnes_necessaires if col in data.columns]
        
        data = data[colonnes_presentes + ['Crypto']]
        data = data[['Date', 'Crypto'] + colonnes_presentes[1:]]
        
        # Suppression des lignes avec des NaN
        data = data.dropna()
        
        # Stockage
        dataframes[ticker] = data
        
        # Sauvegarde individuelle
        filename = f"{output_folder}/{ticker.replace('-', '_')}.csv"
        data.to_csv(filename, index=False)
        
        print(f"   ✅ {ticker} : {len(data)} lignes téléchargées")
        print(f"   💾 Sauvegardé : {filename}")
        
    except Exception as e:
        print(f"   ❌ Erreur pour {ticker} : {str(e)}")

print("\n" + "="*60)
print("🎉 Téléchargement terminé !")

In [ ]:
# Consolidation de toutes les cryptos en un seul DataFrame
if dataframes:
    df_consolidated = pd.concat(dataframes.values(), ignore_index=True)
    df_consolidated = df_consolidated.sort_values(['Date', 'Crypto']).reset_index(drop=True)
    
    # Sauvegarde du fichier consolidé
    consolidated_file = f"{output_folder}/crypto_consolidated.csv"
    df_consolidated.to_csv(consolidated_file, index=False)
    
    print(f"📦 Dataset consolidé créé : {consolidated_file}")
    print(f"📊 Total de lignes : {len(df_consolidated):,}")
    print(f"\n📋 Aperçu des données :")
    display(df_consolidated.head(10))
    
    print(f"\n📊 Statistiques par crypto :")
    print(df_consolidated.groupby('Crypto').size())
else:
    print("❌ Aucune donnée téléchargée")

---

# 2️⃣ FEATURE ENGINEERING : MÉTRIQUES DE BASE

## 🧮 Calcul des variables dérivées

On calcule 3 métriques fondamentales pour l'analyse de volatilité :

In [ ]:
print("⏳ Calcul des métriques de base...\n")

# 1. Volatilité quotidienne (%) : amplitude de fluctuation intraday
df_consolidated['Volatilite_Pct'] = ((df_consolidated['High'] - df_consolidated['Low']) / 
                                      df_consolidated['Close']) * 100

# 2. Variation quotidienne (%) : variation du prix de clôture
df_consolidated['Variation_Pct'] = df_consolidated.groupby('Crypto')['Close'].pct_change() * 100

# 3. Range intraday : amplitude absolue de prix
df_consolidated['Range'] = df_consolidated['High'] - df_consolidated['Low']

print("✅ Métriques calculées : Volatilite_Pct, Variation_Pct, Range")
print(f"\n📊 Aperçu avec nouvelles colonnes :")
display(df_consolidated[['Date', 'Crypto', 'Close', 'Volatilite_Pct', 'Variation_Pct', 'Range']].tail(10))

In [ ]:
# Sauvegarde du dataset avec métriques
metrics_file = f"{output_folder}/crypto_with_metrics.csv"
df_consolidated.to_csv(metrics_file, index=False)

print(f"💾 Fichier avec métriques sauvegardé : {metrics_file}")

---

# 3️⃣ CONSTRUCTION DE LA TABLE DES ÉVÉNEMENTS GÉOPOLITIQUES

## 🌍 30 événements majeurs (2020-2025)

**Critères de sélection** :
- Couverture médiatique internationale
- Impact documenté sur les marchés financiers
- Diversité géographique et thématique

**Attributs** :
- **Date** : Date exacte de l'événement
- **Événement** : Description courte
- **Type** : Catégorie (15 types)
- **Région** : Zone géographique
- **Pays** : Pays impliqués
- **Intensité** : Score 1-10 (impact estimé)

In [ ]:
# Liste exhaustive des événements géopolitiques majeurs
evenements_geopolitiques = [
    # === 2020 ===
    {"Date": "2020-01-03", "Evenement": "Assassinat du général iranien Soleimani par les USA", 
     "Type": "Conflit militaire", "Region": "Moyen-Orient", "Pays": "USA, Iran", "Intensite": 9},
    
    {"Date": "2020-03-11", "Evenement": "OMS déclare la pandémie COVID-19", 
     "Type": "Crise sanitaire mondiale", "Region": "Mondiale", "Pays": "Tous", "Intensite": 10},
    
    {"Date": "2020-03-23", "Evenement": "Lockdown mondial - Krach boursier COVID", 
     "Type": "Crise économique", "Region": "Mondiale", "Pays": "Tous", "Intensite": 10},
    
    {"Date": "2020-05-25", "Evenement": "Mort de George Floyd - Tensions sociales USA", 
     "Type": "Crise sociale", "Region": "Amérique du Nord", "Pays": "USA", "Intensite": 7},
    
    {"Date": "2020-11-03", "Evenement": "Élection présidentielle USA (Biden vs Trump)", 
     "Type": "Élection majeure", "Region": "Amérique du Nord", "Pays": "USA", "Intensite": 8},
    
    # === 2021 ===
    {"Date": "2021-01-06", "Evenement": "Assaut du Capitole américain", 
     "Type": "Crise politique", "Region": "Amérique du Nord", "Pays": "USA", "Intensite": 9},
    
    {"Date": "2021-05-12", "Evenement": "Conflit israélo-palestinien (Gaza)", 
     "Type": "Conflit militaire", "Region": "Moyen-Orient", "Pays": "Israël, Palestine", "Intensite": 8},
    
    {"Date": "2021-08-15", "Evenement": "Chute de Kaboul - Retrait US d'Afghanistan", 
     "Type": "Conflit militaire", "Region": "Asie centrale", "Pays": "USA, Afghanistan", "Intensite": 8},
    
    {"Date": "2021-09-24", "Evenement": "Chine interdit les transactions crypto", 
     "Type": "Régulation crypto", "Region": "Asie", "Pays": "Chine", "Intensite": 9},
    
    {"Date": "2021-11-26", "Evenement": "Variant Omicron COVID détecté", 
     "Type": "Crise sanitaire", "Region": "Mondiale", "Pays": "Tous", "Intensite": 7},
    
    # === 2022 ===
    {"Date": "2022-02-24", "Evenement": "Invasion de l'Ukraine par la Russie", 
     "Type": "Guerre", "Region": "Europe", "Pays": "Russie, Ukraine", "Intensite": 10},
    
    {"Date": "2022-03-01", "Evenement": "Sanctions économiques massives contre la Russie", 
     "Type": "Sanctions économiques", "Region": "Mondiale", "Pays": "Russie, UE, USA", "Intensite": 10},
    
    {"Date": "2022-05-11", "Evenement": "Crash de Terra/LUNA (60 milliards $)", 
     "Type": "Crise crypto", "Region": "Mondiale", "Pays": "Tous", "Intensite": 9},
    
    {"Date": "2022-06-15", "Evenement": "Fed augmente taux de 0.75% (inflation)", 
     "Type": "Politique monétaire", "Region": "Mondiale", "Pays": "USA", "Intensite": 8},
    
    {"Date": "2022-09-21", "Evenement": "Mobilisation partielle en Russie", 
     "Type": "Guerre", "Region": "Europe", "Pays": "Russie", "Intensite": 9},
    
    {"Date": "2022-11-11", "Evenement": "Effondrement de FTX (32 milliards $)", 
     "Type": "Crise crypto", "Region": "Mondiale", "Pays": "Tous", "Intensite": 10},
    
    # === 2023 ===
    {"Date": "2023-03-10", "Evenement": "Faillite Silicon Valley Bank (SVB)", 
     "Type": "Crise bancaire", "Region": "Amérique du Nord", "Pays": "USA", "Intensite": 9},
    
    {"Date": "2023-03-19", "Evenement": "Credit Suisse racheté par UBS (crise bancaire)", 
     "Type": "Crise bancaire", "Region": "Europe", "Pays": "Suisse", "Intensite": 8},
    
    {"Date": "2023-06-04", "Evenement": "SEC poursuit Binance et Coinbase", 
     "Type": "Régulation crypto", "Region": "Amérique du Nord", "Pays": "USA", "Intensite": 8},
    
    {"Date": "2023-10-07", "Evenement": "Attaque du Hamas contre Israël - Début guerre Gaza", 
     "Type": "Conflit militaire", "Region": "Moyen-Orient", "Pays": "Israël, Hamas", "Intensite": 10},
    
    {"Date": "2023-11-22", "Evenement": "Sam Altman (OpenAI) évincé puis réintégré", 
     "Type": "Crise tech", "Region": "Mondiale", "Pays": "USA", "Intensite": 6},
    
    # === 2024 ===
    {"Date": "2024-01-10", "Evenement": "Approbation des ETF Bitcoin spot par la SEC", 
     "Type": "Régulation crypto", "Region": "Mondiale", "Pays": "USA", "Intensite": 9},
    
    {"Date": "2024-03-01", "Evenement": "Tensions USA-Chine sur Taiwan (manœuvres militaires)", 
     "Type": "Tensions géopolitiques", "Region": "Asie", "Pays": "USA, Chine, Taiwan", "Intensite": 8},
    
    {"Date": "2024-04-13", "Evenement": "Attaque iranienne contre Israël (missiles/drones)", 
     "Type": "Conflit militaire", "Region": "Moyen-Orient", "Pays": "Iran, Israël", "Intensite": 9},
    
    {"Date": "2024-04-19", "Evenement": "Bitcoin Halving (réduction récompense minage)", 
     "Type": "Événement crypto technique", "Region": "Mondiale", "Pays": "Tous", "Intensite": 7},
    
    {"Date": "2024-07-13", "Evenement": "Tentative d'assassinat de Donald Trump", 
     "Type": "Crise politique", "Region": "Amérique du Nord", "Pays": "USA", "Intensite": 8},
    
    {"Date": "2024-08-05", "Evenement": "Krach boursier mondial (VIX +65%, Nikkei -12%)", 
     "Type": "Crise financière", "Region": "Mondiale", "Pays": "Tous", "Intensite": 9},
    
    {"Date": "2024-10-01", "Evenement": "Opération terrestre israélienne au Liban", 
     "Type": "Conflit militaire", "Region": "Moyen-Orient", "Pays": "Israël, Liban", "Intensite": 8},
    
    {"Date": "2024-11-05", "Evenement": "Élection présidentielle USA (Trump élu)", 
     "Type": "Élection majeure", "Region": "Amérique du Nord", "Pays": "USA", "Intensite": 9},
    
    # === 2025 ===
    {"Date": "2025-01-20", "Evenement": "Investiture de Donald Trump", 
     "Type": "Transition politique", "Region": "Amérique du Nord", "Pays": "USA", "Intensite": 7},
]

# Création du DataFrame
df_evenements = pd.DataFrame(evenements_geopolitiques)
df_evenements['Date'] = pd.to_datetime(df_evenements['Date'])

# Ajout de colonnes temporelles
df_evenements['Annee'] = df_evenements['Date'].dt.year
df_evenements['Mois'] = df_evenements['Date'].dt.month
df_evenements['Trimestre'] = df_evenements['Date'].dt.quarter

# Tri chronologique
df_evenements = df_evenements.sort_values('Date').reset_index(drop=True)

print(f"✅ Table des événements créée : {len(df_evenements)} événements")
print(f"\n📋 Aperçu :")
display(df_evenements.head(10))

In [ ]:
# Statistiques descriptives
print("📊 Répartition par type d'événement :")
print(df_evenements['Type'].value_counts())

print("\n📊 Répartition par région :")
print(df_evenements['Region'].value_counts())

print("\n📊 Répartition par année :")
print(df_evenements['Annee'].value_counts().sort_index())

print("\n📊 Intensité moyenne :")
print(f"Moyenne : {df_evenements['Intensite'].mean():.1f}/10")
print(f"Médiane : {df_evenements['Intensite'].median():.0f}/10")

In [ ]:
# Sauvegarde de la table des événements
evenements_file = f"{output_folder}/evenements_geopolitiques.csv"
df_evenements.to_csv(evenements_file, index=False)

print(f"💾 Table des événements sauvegardée : {evenements_file}")

---

# 4️⃣ CRÉATION DU CALENDRIER ENRICHI

## 📅 Fenêtres temporelles autour des événements

Pour chaque date du calendrier (2020-2025), on calcule :
- **Jours_Depuis_Evenement** : Distance en jours depuis le dernier événement
- **Jours_Avant_Evenement** : Distance en jours jusqu'au prochain événement
- **Periode_Evenement** : Classification (Avant 1-7j, Jour J, Après 1-7j, etc.)
- **Evenement_Proche** : Nom de l'événement le plus proche

Cette structure permet l'analyse temporelle ±30j autour des chocs géopolitiques.

In [ ]:
# Création du calendrier complet
date_range = pd.date_range(start='2020-01-01', end='2025-12-31', freq='D')
df_calendrier = pd.DataFrame({'Date': date_range})

# Extraction des composantes temporelles
df_calendrier['Annee'] = df_calendrier['Date'].dt.year
df_calendrier['Mois'] = df_calendrier['Date'].dt.month
df_calendrier['Jour'] = df_calendrier['Date'].dt.day
df_calendrier['Trimestre'] = df_calendrier['Date'].dt.quarter
df_calendrier['Semaine'] = df_calendrier['Date'].dt.isocalendar().week
df_calendrier['Jour_Semaine'] = df_calendrier['Date'].dt.dayofweek + 1
df_calendrier['Nom_Jour'] = df_calendrier['Date'].dt.day_name()
df_calendrier['Nom_Mois'] = df_calendrier['Date'].dt.month_name()

print(f"📅 Calendrier créé : {len(df_calendrier)} jours")
print(f"📊 Période : {df_calendrier['Date'].min()} → {df_calendrier['Date'].max()}")

In [ ]:
# Fonction de classification des périodes (corrigée pour inclure "Avant")
def classifier_periode_corrigee(row, dates_evenements):
    """
    Classifie une date selon sa proximité avec les événements.
    Calcule la distance AVANT (prochain événement) et APRÈS (dernier événement).
    """
    date = row['Date']
    
    # Distance depuis le dernier événement passé
    passes = [(date - evt).days for evt in dates_evenements if evt <= date]
    # Distance jusqu'au prochain événement futur
    futurs = [(evt - date).days for evt in dates_evenements if evt > date]
    
    jours_depuis = min(passes) if passes else None
    jours_avant = min(futurs) if futurs else None
    
    # Priorité : Jour J d'abord
    if jours_depuis == 0:
        return "Jour de l'événement", 0

    # Ensuite : période AVANT (prochain événement dans le futur)
    if jours_avant is not None:
        if 1 <= jours_avant <= 7:
            return "Avant événement (1-7j)", -jours_avant
        elif 8 <= jours_avant <= 30:
            return "Avant événement (8-30j)", -jours_avant

    # Ensuite : période APRÈS (dernier événement dans le passé)
    if jours_depuis is not None:
        if 1 <= jours_depuis <= 7:
            return "Après événement (1-7j)", jours_depuis
        elif 8 <= jours_depuis <= 30:
            return "Après événement (8-30j)", jours_depuis

    return "Hors événement", jours_depuis if jours_depuis else 999

In [ ]:
# Application de la classification
print("⏳ Classification des périodes (peut prendre 1-2 min)...\n")

dates_evenements = sorted(df_evenements['Date'].tolist())

resultats = df_calendrier.apply(lambda row: classifier_periode_corrigee(row, dates_evenements), axis=1)
df_calendrier['Periode_Evenement'] = resultats.apply(lambda x: x[0])
df_calendrier['Jours_Depuis_Evenement'] = resultats.apply(lambda x: x[1])

# Indicateur binaire : période d'influence (±30 jours)
df_calendrier['Est_Periode_Evenement'] = df_calendrier['Jours_Depuis_Evenement'].apply(
    lambda x: 1 if x is not None and -30 <= x <= 30 else 0
)

print("✅ Classification terminée !")
print("\n📊 Répartition des périodes :")
print(df_calendrier['Periode_Evenement'].value_counts())

In [ ]:
# Enrichissement avec les détails des événements
def trouver_evenement_proche(date, df_evt):
    """
    Trouve l'événement le plus proche dans la fenêtre ±30 jours.
    """
    evenements_proches = df_evt[
        (df_evt['Date'] >= date - pd.Timedelta(days=30)) & 
        (df_evt['Date'] <= date + pd.Timedelta(days=30))
    ]
    
    if len(evenements_proches) == 0:
        return None, None, None, None, None
    
    # Prendre l'événement le plus proche
    evenements_proches['Distance'] = abs((evenements_proches['Date'] - date).dt.days)
    evt_plus_proche = evenements_proches.sort_values('Distance').iloc[0]
    
    return (
        evt_plus_proche['Evenement'],
        evt_plus_proche['Type'],
        evt_plus_proche['Region'],
        evt_plus_proche['Intensite'],
        evt_plus_proche['Date']
    )

print("⏳ Enrichissement avec détails événements...\n")

df_calendrier[['Evenement_Proche', 'Type_Evenement', 'Region_Evenement', 
               'Intensite_Evenement', 'Date_Evenement_Proche']] = df_calendrier['Date'].apply(
    lambda x: pd.Series(trouver_evenement_proche(x, df_evenements))
)

print("✅ Enrichissement terminé !")

In [ ]:
# Sauvegarde du calendrier enrichi
calendrier_file = f"{output_folder}/calendrier_enrichi.csv"
df_calendrier.to_csv(calendrier_file, index=False)

print(f"💾 Calendrier enrichi sauvegardé : {calendrier_file}")
print(f"\n📊 Statistiques :")
print(f"  • Jours en période d'événement (±30j) : {df_calendrier['Est_Periode_Evenement'].sum()}")
print(f"  • Jours hors événement : {len(df_calendrier) - df_calendrier['Est_Periode_Evenement'].sum()}")
print(f"  • Pourcentage couvert : {df_calendrier['Est_Periode_Evenement'].mean()*100:.1f}%")

---

# 5️⃣ FUSION ET FEATURE ENGINEERING AVANCÉ

## 🔗 Fusion des données crypto avec le calendrier enrichi

On fusionne `df_consolidated` (données crypto) avec `df_calendrier` (périodes) 
pour obtenir un dataset final complet avec toutes les variables nécessaires.

In [ ]:
print("🔄 Fusion des données crypto avec le calendrier enrichi...\n")

# Fusion sur la date
df_analyse = df_consolidated.merge(df_calendrier, on='Date', how='left')

print(f"✅ Fusion réalisée : {len(df_analyse):,} lignes")
print(f"📊 Colonnes disponibles : {len(df_analyse.columns)}")

## 📈 Moyennes mobiles (lissage de tendance)

In [ ]:
print("⏳ Calcul des moyennes mobiles (MM7, MM30)...\n")

# Tri pour les calculs de fenêtres glissantes
df_analyse = df_analyse.sort_values(['Crypto', 'Date']).reset_index(drop=True)

# Moyennes mobiles sur 7 et 30 jours (prix)
for window in [7, 30]:
    df_analyse[f'Prix_MM{window}'] = df_analyse.groupby('Crypto')['Close'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

# Moyennes mobiles sur 7 et 30 jours (volume)
for window in [7, 30]:
    df_analyse[f'Volume_MM{window}'] = df_analyse.groupby('Crypto')['Volume'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

print("✅ Moyennes mobiles calculées : Prix_MM7, Prix_MM30, Volume_MM7, Volume_MM30")

## 🔥 Détection des chocs (volatilité et volume anormaux)

In [ ]:
print("⏳ Calcul des chocs de volatilité et volume...\n")

# Volatilité sur fenêtre glissante
for window in [7, 30]:
    df_analyse[f'Volatilite_MM{window}'] = df_analyse.groupby('Crypto')['Volatilite_Pct'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

# Choc de volatilité : écart relatif par rapport à MM30
df_analyse['Choc_Volatilite'] = (
    (df_analyse['Volatilite_Pct'] - df_analyse['Volatilite_MM30']) / 
    df_analyse['Volatilite_MM30']
) * 100

# Choc de volume : écart relatif par rapport à MM30
df_analyse['Choc_Volume'] = (
    (df_analyse['Volume'] - df_analyse['Volume_MM30']) / 
    df_analyse['Volume_MM30']
) * 100

print("✅ Chocs calculés : Choc_Volatilite, Choc_Volume")

## 📊 Performance depuis l'événement

In [ ]:
print("⏳ Calcul de la performance depuis événement (peut prendre 2-3 min)...\n")

def calculer_performance_depuis_evenement(row, df_full):
    """
    Calcule la variation % du prix depuis le jour de l'événement.
    """
    if pd.isna(row['Date_Evenement_Proche']) or row['Jours_Depuis_Evenement'] <= 0:
        return None
    
    # Trouver le prix au jour de l'événement
    mask = (
        (df_full['Crypto'] == row['Crypto']) & 
        (df_full['Date'] == row['Date_Evenement_Proche'])
    )
    
    if mask.sum() == 0:
        return None
    
    prix_evenement = df_full.loc[mask, 'Close'].iloc[0]
    return ((row['Close'] - prix_evenement) / prix_evenement) * 100

df_analyse['Performance_Depuis_Evenement_%'] = df_analyse.apply(
    lambda row: calculer_performance_depuis_evenement(row, df_analyse), axis=1
)

print("✅ Performance calculée : Performance_Depuis_Evenement_%")

## 🏷️ Catégorisation (pour visuels Power BI)

In [ ]:
print("⏳ Catégorisation de la volatilité et intensité...\n")

# Catégorie de volatilité
df_analyse['Categorie_Volatilite'] = pd.cut(
    df_analyse['Volatilite_Pct'],
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=['Faible (<2%)', 'Modérée (2-5%)', 'Élevée (5-10%)', 'Extrême (>10%)']
)

# Catégorie d'intensité d'événement
df_analyse['Categorie_Intensite'] = pd.cut(
    df_analyse['Intensite_Evenement'],
    bins=[0, 6, 8, 10],
    labels=['Faible (1-6)', 'Moyenne (7-8)', 'Forte (9-10)'],
    include_lowest=True
)

print("✅ Catégories créées : Categorie_Volatilite, Categorie_Intensite")

---

# 6️⃣ NETTOYAGE FINAL ET EXPORT POUR POWER BI

## 🧹 Gestion des valeurs manquantes

In [ ]:
print("🧹 Nettoyage des valeurs manquantes...\n")
print(f"❌ NaN totaux avant nettoyage : {df_analyse.isnull().sum().sum()}")

# Colonnes numériques : remplacer NaN par 0
colonnes_numeriques_zero = [
    'Open', 'High', 'Low', 'Close', 'Volume', 
    'Volatilite_Pct', 'Variation_Pct', 'Range',
    'Prix_MM7', 'Prix_MM30', 'Volume_MM7', 'Volume_MM30',
    'Volatilite_MM7', 'Volatilite_MM30',
    'Performance_Depuis_Evenement_%', 'Choc_Volatilite', 'Choc_Volume'
]

for col in colonnes_numeriques_zero:
    if col in df_analyse.columns:
        df_analyse[col] = pd.to_numeric(df_analyse[col], errors='coerce').fillna(0)

# Intensité : remplacer NaN par 0
if 'Intensite_Evenement' in df_analyse.columns:
    df_analyse['Intensite_Evenement'] = pd.to_numeric(df_analyse['Intensite_Evenement'], errors='coerce').fillna(0)

# Jours_Depuis_Evenement : remplacer NaN par 999 (très loin)
if 'Jours_Depuis_Evenement' in df_analyse.columns:
    df_analyse['Jours_Depuis_Evenement'] = pd.to_numeric(df_analyse['Jours_Depuis_Evenement'], errors='coerce').fillna(999)

# Colonnes texte : remplacer NaN par "Non défini"
colonnes_texte = ['Evenement_Proche', 'Type_Evenement', 'Region_Evenement', 
                  'Periode_Evenement', 'Categorie_Volatilite', 'Categorie_Intensite']

for col in colonnes_texte:
    if col in df_analyse.columns:
        df_analyse[col] = df_analyse[col].fillna("Non défini")

print(f"✅ NaN restants après nettoyage : {df_analyse.isnull().sum().sum()}")

## 🔢 Conversion et arrondissement

In [ ]:
print("⏳ Conversion des types de données...\n")

# Dates
df_analyse['Date'] = pd.to_datetime(df_analyse['Date'], errors='coerce')
if 'Date_Evenement_Proche' in df_analyse.columns:
    df_analyse['Date_Evenement_Proche'] = pd.to_datetime(df_analyse['Date_Evenement_Proche'], errors='coerce')

# Entiers
colonnes_int = ['Annee', 'Mois', 'Jour', 'Trimestre', 'Semaine', 'Jour_Semaine', 'Est_Periode_Evenement']
for col in colonnes_int:
    if col in df_analyse.columns:
        df_analyse[col] = pd.to_numeric(df_analyse[col], errors='coerce').fillna(0).astype(int)

# Arrondissement des nombres (4 décimales max)
colonnes_a_arrondir = ['Open', 'High', 'Low', 'Close', 'Volatilite_Pct', 'Variation_Pct',
                       'Prix_MM7', 'Prix_MM30', 'Volatilite_MM7', 'Volatilite_MM30',
                       'Performance_Depuis_Evenement_%', 'Choc_Volatilite', 'Choc_Volume']

for col in colonnes_a_arrondir:
    if col in df_analyse.columns:
        df_analyse[col] = df_analyse[col].round(4)

# Volume en entiers
if 'Volume' in df_analyse.columns:
    df_analyse['Volume'] = df_analyse['Volume'].round(0).astype('int64')
if 'Volume_MM7' in df_analyse.columns:
    df_analyse['Volume_MM7'] = df_analyse['Volume_MM7'].round(0).astype('int64')
if 'Volume_MM30' in df_analyse.columns:
    df_analyse['Volume_MM30'] = df_analyse['Volume_MM30'].round(0).astype('int64')

print("✅ Types de données optimisés")

## 💾 Export final en Excel (pour Power BI)

In [ ]:
fichier_final = f"{output_folder}/dataset_powerbi_clean.xlsx"

print("⏳ Export en Excel...\n")

with pd.ExcelWriter(fichier_final, engine='openpyxl') as writer:
    df_analyse.to_excel(writer, sheet_name='Donnees', index=False)

print(f"✅ Dataset final créé : {fichier_final}")
print(f"📊 Dimensions : {df_analyse.shape[0]:,} lignes × {df_analyse.shape[1]} colonnes")
print(f"✓ NaN restants : {df_analyse.isnull().sum().sum()}")

In [ ]:
# Aperçu final
print("\n📋 Aperçu du dataset final :")
display(df_analyse[['Date', 'Crypto', 'Close', 'Volatilite_Pct', 'Periode_Evenement', 
                    'Evenement_Proche', 'Intensite_Evenement']].tail(10))

In [ ]:
# Statistiques finales
print("\n" + "="*60)
print("📊 STATISTIQUES FINALES\n")

print("1️⃣ Volatilité moyenne par crypto :")
print(df_analyse.groupby('Crypto')['Volatilite_Pct'].agg(['mean', 'std', 'max']).round(2))

print("\n2️⃣ Volatilité : Période événement vs Hors événement :")
print(df_analyse.groupby('Est_Periode_Evenement')['Volatilite_Pct'].agg(['mean', 'median']).round(2))

print("\n3️⃣ Volume moyen : Période événement vs Hors événement :")
print(df_analyse.groupby('Est_Periode_Evenement')['Volume'].agg(['mean', 'median']))

print("\n" + "="*60)

---

# 7️⃣ CRÉATION DE LA TABLE GRANDES PUISSANCES

## 🌐 Classification USA / Chine / Europe / Moyen-Orient

Pour l'analyse géopolitique (Questions 6 & 7), on crée une table séparée 
qui classe chaque événement selon la puissance géopolitique impliquée.

In [ ]:
# Classification par puissance géopolitique
grandes_puissances = [
    # USA
    {"Evenement": "Assassinat du général iranien Soleimani par les USA", "Puissance": "USA"},
    {"Evenement": "Élection présidentielle USA (Biden vs Trump)", "Puissance": "USA"},
    {"Evenement": "Assaut du Capitole américain", "Puissance": "USA"},
    {"Evenement": "Fed augmente taux de 0.75% (inflation)", "Puissance": "USA"},
    {"Evenement": "SEC poursuit Binance et Coinbase", "Puissance": "USA"},
    {"Evenement": "Approbation des ETF Bitcoin spot par la SEC", "Puissance": "USA"},
    {"Evenement": "Tentative d'assassinat de Donald Trump", "Puissance": "USA"},
    {"Evenement": "Élection présidentielle USA (Trump élu)", "Puissance": "USA"},
    {"Evenement": "Investiture de Donald Trump", "Puissance": "USA"},
    {"Evenement": "Faillite Silicon Valley Bank (SVB)", "Puissance": "USA"},
    {"Evenement": "Krach boursier mondial (VIX +65%, Nikkei -12%)", "Puissance": "USA"},
    {"Evenement": "Chute de Kaboul - Retrait US d'Afghanistan", "Puissance": "USA"},
    {"Evenement": "Mort de George Floyd - Tensions sociales USA", "Puissance": "USA"},
    {"Evenement": "Sam Altman (OpenAI) évincé puis réintégré", "Puissance": "USA"},

    # CHINE
    {"Evenement": "Chine interdit les transactions crypto", "Puissance": "Chine"},
    {"Evenement": "Tensions USA-Chine sur Taiwan (manœuvres militaires)", "Puissance": "Chine"},

    # EUROPE / RUSSIE
    {"Evenement": "Invasion de l'Ukraine par la Russie", "Puissance": "Europe/Russie"},
    {"Evenement": "Sanctions économiques massives contre la Russie", "Puissance": "Europe/Russie"},
    {"Evenement": "Mobilisation partielle en Russie", "Puissance": "Europe/Russie"},
    {"Evenement": "Credit Suisse racheté par UBS (crise bancaire)", "Puissance": "Europe/Russie"},

    # MOYEN-ORIENT
    {"Evenement": "Conflit israélo-palestinien (Gaza)", "Puissance": "Moyen-Orient"},
    {"Evenement": "Attaque du Hamas contre Israël - Début guerre Gaza", "Puissance": "Moyen-Orient"},
    {"Evenement": "Attaque iranienne contre Israël (missiles/drones)", "Puissance": "Moyen-Orient"},
    {"Evenement": "Opération terrestre israélienne au Liban", "Puissance": "Moyen-Orient"},

    # MONDIAL
    {"Evenement": "OMS déclare la pandémie COVID-19", "Puissance": "Mondial"},
    {"Evenement": "Lockdown mondial - Krach boursier COVID", "Puissance": "Mondial"},
    {"Evenement": "Variant Omicron COVID détecté", "Puissance": "Mondial"},
    {"Evenement": "Crash de Terra/LUNA (60 milliards $)", "Puissance": "Mondial"},
    {"Evenement": "Effondrement de FTX (32 milliards $)", "Puissance": "Mondial"},
    {"Evenement": "Bitcoin Halving (réduction récompense minage)", "Puissance": "Mondial"},
]

# Création du DataFrame
df_puissances = pd.DataFrame(grandes_puissances)

# Fusion avec les événements pour récupérer les dates et intensités
df_puissances_enrichi = df_puissances.merge(
    df_evenements[['Evenement', 'Date', 'Type', 'Intensite', 'Region']],
    on='Evenement',
    how='left'
)

# Conversion date
df_puissances_enrichi['Date'] = pd.to_datetime(df_puissances_enrichi['Date'])
df_puissances_enrichi['Annee'] = df_puissances_enrichi['Date'].dt.year
df_puissances_enrichi['Intensite'] = df_puissances_enrichi['Intensite'].fillna(0).astype(int)

print("✅ Table Grandes Puissances créée !")
print(f"📊 Nombre d'entrées : {len(df_puissances_enrichi)}")
print(f"\n📋 Répartition par puissance :")
print(df_puissances_enrichi['Puissance'].value_counts())

In [ ]:
# Export Excel
puissances_file = f"{output_folder}/grandes_puissances.xlsx"

with pd.ExcelWriter(puissances_file, engine='openpyxl') as writer:
    df_puissances_enrichi.to_excel(writer, sheet_name='Grandes_Puissances', index=False)

print(f"💾 Fichier sauvegardé : {puissances_file}")
print(f"\n📋 Aperçu :")
display(df_puissances_enrichi[['Date', 'Evenement', 'Puissance', 'Intensite']].head(10))

---

# ✅ RÉSUMÉ FINAL

## 📁 Fichiers créés dans `crypto_data/`

| Fichier | Description | Taille |
|---------|-------------|--------|
| `BTC_USD.csv` | Données Bitcoin | ~2 192 lignes |
| `ETH_USD.csv` | Données Ethereum | ~2 192 lignes |
| `DOGE_USD.csv` | Données Dogecoin | ~2 192 lignes |
| `USDC_USD.csv` | Données USD Coin | ~2 192 lignes |
| `crypto_consolidated.csv` | Toutes cryptos consolidées | ~8 768 lignes |
| `crypto_with_metrics.csv` | Avec volatilité, variation | ~8 768 lignes |
| `evenements_geopolitiques.csv` | 30 événements majeurs | 30 lignes |
| `calendrier_enrichi.csv` | Calendrier avec fenêtres ±30j | ~2 192 lignes |
| `grandes_puissances.xlsx` | Classification géopolitique | 30 lignes |
| **`dataset_powerbi_clean.xlsx`** | **FICHIER PRINCIPAL POWER BI** | **8 768 × 38** |

---

## 🎯 Prochaines étapes

1. ✅ Ouvrir Power BI Desktop
2. ✅ Importer `dataset_powerbi_clean.xlsx`
3. ✅ Importer `grandes_puissances.xlsx`
4. ✅ Créer la relation entre les deux tables
5. ✅ Créer les 23 mesures DAX
6. ✅ Construire les 5 dashboards

---

**🎉 Pipeline de données terminé avec succès !**